In [1]:
import random

import asyncio
import nest_asyncio

import networkx as nx


from src.utils import Loader
from src.metrics import utility_gap
from src.diffusion_models import estimate_cascade_by_community

In [2]:
random.seed(42)
nest_asyncio.apply()

In [3]:
from pathlib import Path

paths_to_networks = Path('src/data/synthetic/networks')

## Simple Barbasi-Albert Graph

In [4]:
k = 10  # number of seeds to select
alpha = 0  # inequality-aversion parameter
p = 0.1  # edge activation probability
num_sims = 100  # number of simulations

In [5]:
from src import kempe_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')

    seeds = kempe_greedy(
        graph=loaded_graph,
        k=k,
        probability=p,
        num_simulations=num_sims,
    )
    print(f'Final seeds: {seeds}')

    final_frac = estimate_cascade_by_community(
        graph=loaded_graph,
        seeds=seeds,
        probability=0.1,
        num_simulations=500,
    )
    print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in final_frac.items()})}')
    print(f'Utility Gap: {round(utility_gap(final_frac), 2)}')


asyncio.run(main())

Selecting seeds: 100%|██████████| 10/10 [00:57<00:00,  5.80s/it, seeds=10]

Final seeds: {0, 4, 5, 167, 10, 75, 14, 308, 27, 31}
Expected influenced fraction per community: {0: 0.1, 1: 0.06, 2: 0.07, 3: 0.1, 4: 0.06, 5: 0.09, 6: 0.03, 7: 0.06, 8: 0.02, 9: 0.09, 10: 0.06, 11: 0.05, 12: 0.05, 13: 0.07, 14: 0.03, 15: 0.01, 16: 0.03}
Utility Gap: 8.77


In [6]:
from src import welfare_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')

    communities = set(nx.get_node_attributes(loaded_graph, 'community').values())

    seeds = welfare_greedy(
        graph=loaded_graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )

    print(f'Selected seed nodes: {seeds}')

    final_frac = estimate_cascade_by_community(
        graph=loaded_graph,
        seeds=seeds,
        probability=0.1,
        num_simulations=500,
    )
    print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in final_frac.items()})}')
    print(f'Utility Gap: {round(utility_gap(final_frac), 2)}')


asyncio.run(main())

Selecting seeds: 100%|██████████| 10/10 [01:02<00:00,  6.26s/it, seeds=10, influenced={0: 0.1, 1: 0.06, 2: 0.06, 3: 0.08, 4: 0.06, 5: 0.07, 6: 0.05, 7: 0.05, 8: 0.05, 9: 0.06, 10: 0.05, 11: 0.05, 12: 0.07, 13: 0.06, 14: 0.05, 15: 0.05, 16: 0.05}]

Selected seed nodes: {0, 197, 646, 263, 267, 45, 207, 470, 826, 381}
Expected influenced fraction per community: {0: 0.1, 1: 0.06, 2: 0.06, 3: 0.08, 4: 0.05, 5: 0.07, 6: 0.05, 7: 0.06, 8: 0.05, 9: 0.06, 10: 0.04, 11: 0.05, 12: 0.07, 13: 0.06, 14: 0.05, 15: 0.05, 16: 0.05}
Utility Gap: 5.98
